In [1]:
import dac
import glob

model_path = dac.utils.download(model_type="16khz")
dac = dac.DAC.load(model_path).eval()

/home/harry/diarisation/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [5]:
from nanodrz.data import libritts_test
wav_files = [u.file_url for u in libritts_test()[0].utts]

In [8]:
import torch
import torchaudio

from nanodrz.utils import play
audio, sr = torchaudio.load(wav_files[0])

print(sr)
resampled_audio = torchaudio.transforms.Resample(sr, 16000)(audio)
print(dac.sample_rate)
play(resampled_audio, 16000)

24000
16000


In [48]:
resampled_audio = dac.preprocess(resampled_audio, 16000)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(resampled_audio.shape)
x = dac.to(device).encode(audio_data=resampled_audio[None].to(device))

torch.Size([1, 90880])


In [50]:
stacked_code_embs = [dac.quantizer.quantizers[i].codebook(x[1][:,i]) for i in range(len(dac.quantizer.quantizers))]
stacked_code_embs = torch.cat(stacked_code_embs, dim=-1)
stacked_code_embs.shape

torch.Size([1, 284, 96])

In [51]:
from torch import nn

convcompress = nn.Sequential(
    nn.Conv1d(1024, 1024//2, kernel_size=3),
    nn.GELU(),
    nn.Conv1d(1024//2, 256, kernel_size=3),
    nn.GELU(),
)

convcompress(x[0]).shape

torch.Size([1, 1024, 284])


torch.Size([1, 256, 280])

In [28]:
print("Dac Z Compression", resampled_audio.numel() / x[0].numel())
mel = torchaudio.transforms.MelSpectrogram(n_fft=1024, win_length=256, hop_length=256, n_mels=80)(resampled_audio)
print(mel.shape, audio.shape)
print("Mel Compression", resampled_audio.numel() / mel.numel())

Dac Z Compression 0.3125
torch.Size([1, 80, 356]) torch.Size([1, 135841])
Mel Compression 3.191011235955056


In [ ]:
print(x[0].shape, resampled_audio.shape[-1]/x[0].shape[-1])

In [ ]:
import torch

In [ ]:
resampled_audio = dac.preprocess(torch.rand(1, 1, 16000*3).cuda(), 16000)
dac.cuda().encode(audio_data=resampled_audio)[0].shape

In [ ]:
16000*3/150